In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def get_steam_top_sellers():
    driver = webdriver.Chrome()
    driver.get('https://store.steampowered.com/charts/topsellers/TW/2026-05-12')
    
    # 等待40秒讓頁面完全載入
    time.sleep(40)
    
    # 點擊"查看所有100項"按鈕
    view_all_button = driver.find_element(By.CLASS_NAME, 'DialogButton._DialogLayout.Primary.Focusable')
    view_all_button.click()
    
    # 再等待載入
    time.sleep(3)
    
    games = []
    # 找到所有遊戲標題和價格
    titles = driver.find_elements(By.CLASS_NAME, '_1n_4-zvf0n4aqGEksbgW9N')
    prices = driver.find_elements(By.CLASS_NAME, '_3j4dI1yA7cRfCvK8h406OB')
    
    # 將標題和價格配對
    for title, price in zip(titles, prices):
        games.append({
            'title': title.text,
            'price': price.text
        })
    
    driver.quit()
    
    return pd.DataFrame(games)

# 執行爬蟲
df = get_steam_top_sellers()

# 儲存成 CSV
df.to_csv('steam_top_sellers.csv', index=False, encoding='utf-8-sig')

# 顯示 DataFrame(表格樣式)
df

,title,price
0,Forza Horizon 6,"NT$ 1,990.00"
1,PUBG: BATTLEGROUNDS,免費遊玩
2,Subnautica 2,NT$ 699.00
3,燕雲十六聲,免費遊玩
4,Counter-Strike 2,免費遊玩
...,...,...
95,龍胤立志傳,NT$ 328.00
96,Gray Zone Warfare,NT$ 891.00
97,《NBA 2K26》名人堂通行證：第7季,NT$ 590.00
98,Devil May Cry 4 Special Edition,NT$ 140.00


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from tqdm import tqdm
import pandas as pd
import time


def get_steam_top_sellers():
    # 所有要爬取的日期(2024 ~ 2026-05)
    dates = [
        # 2024
        "2024-1-2", "2024-1-9", "2024-1-16", "2024-1-23", "2024-1-30",
        "2024-2-6", "2024-2-13", "2024-2-20", "2024-2-27",
        "2024-3-5", "2024-3-12", "2024-3-19", "2024-3-26",
        "2024-4-2", "2024-4-9", "2024-4-16", "2024-4-23", "2024-4-30",
        
        "2024-5-7", "2024-5-14", "2024-5-21", "2024-5-28",
        "2024-6-4", "2024-6-11", "2024-6-18", "2024-6-25",
        "2024-7-2", "2024-7-9", "2024-7-16", "2024-7-23", "2024-7-30",
        "2024-8-6", "2024-8-13", "2024-8-20", "2024-8-27",
        "2024-9-3", "2024-9-10", "2024-9-17", "2024-9-24",
        "2024-10-1", "2024-10-8", "2024-10-15", "2024-10-22", "2024-10-29",
        "2024-11-5", "2024-11-12", "2024-11-19", "2024-11-26",
        "2024-12-3", "2024-12-10", "2024-12-17", "2024-12-24", "2024-12-31",
        # 2025
        "2025-1-7", "2025-1-14", "2025-1-21", "2025-1-28",
        "2025-2-4", "2025-2-11", "2025-2-18", "2025-2-25",
        "2025-3-4", "2025-3-11", "2025-3-18", "2025-3-25",
        "2025-4-1", "2025-4-8", "2025-4-15", "2025-4-22", "2025-4-29",
        "2025-5-6", "2025-5-13", "2025-5-20", "2025-5-27",
        "2025-6-3", "2025-6-10", "2025-6-17", "2025-6-24",
        "2025-7-1", "2025-7-8", "2025-7-15", "2025-7-22", "2025-7-29",
        "2025-8-5", "2025-8-12", "2025-8-19", "2025-8-26",
        "2025-9-2", "2025-9-9", "2025-9-16", "2025-9-23", "2025-9-30",
        "2025-10-7", "2025-10-14", "2025-10-21", "2025-10-28",
        "2025-11-4", "2025-11-11", "2025-11-18", "2025-11-25",
        "2025-12-2", "2025-12-9", "2025-12-16", "2025-12-23", "2025-12-30",
        # 2026
        "2026-1-6", "2026-1-13", "2026-1-20", "2026-1-27",
        "2026-2-3", "2026-2-10", "2026-2-17", "2026-2-24",
        "2026-3-3", "2026-3-10", "2026-3-17", "2026-3-24", "2026-3-31",
        "2026-4-7", "2026-4-14", "2026-4-21", "2026-4-28",
        "2026-5-5", "2026-5-12", "2026-5-19"
    ]
    
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 20)
    games = []
    
    for i, date in enumerate(tqdm(dates, desc="爬取進度", ncols=100)):
        driver.get(f'https://store.steampowered.com/charts/topsellers/TW/{date}')
        
        # 第一次等久一點讓 Chrome 完全初始化
        if i == 0:
            time.sleep(50)
        else:
            time.sleep(3)
        
        # === 關鍵 1:確保點到「查看全部」按鈕 ===
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            view_all_button = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.DialogButton[type='button']"))
            )
            driver.execute_script("arguments[0].click();", view_all_button)
            time.sleep(3)
        except:
            tqdm.write(f"[警告] 找不到「查看全部」按鈕 - {date}")
        
        # === 關鍵 2:多次滾動確保 100 筆全部 Lazy Load 完成 ===
        last_count = 0
        for _ in range(10):  # 最多滾 10 次
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
            
            rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')
            current_count = len(rows)
            
            if current_count >= 100:
                break  # 已經 100 筆,停止滾動
            if current_count == last_count:
                # 連續兩次數量沒變,代表載完了
                break
            last_count = current_count
        
        # === 關鍵 3:逐列抓取,避免 zip() 截斷 ===
        rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')
        week_count = 0
        
        for row in rows:
            # 抓標題
            try:
                title = row.find_element(By.CLASS_NAME, '_1n_4-zvf0n4aqGEksbgW9N').text
            except:
                continue  # 沒標題就跳過這一列
            
            # 抓價格(找不到代表是免費)
            try:
                price = row.find_element(By.CLASS_NAME, '_3j4dI1yA7cRfCvK8h406OB').text
            except:
                price = '免費'
            
            games.append({
                'date': date,
                'title': title,
                'price': price
            })
            week_count += 1
        
        # 用 tqdm.write() 印,不會破壞進度條
        tqdm.write(f"{date} 完成 - {week_count} 筆")
    
    driver.quit()
    
    return pd.DataFrame(games)


# 執行爬蟲
df = get_steam_top_sellers()

# 儲存成 CSV
df.to_csv('steam_top_sellers.csv', index=False, encoding='utf-8-sig')

# 顯示結果(DataFrame 格式)
print(f"\n總共爬取 {len(df)} 筆資料,已儲存至 steam_top_sellers.csv\n")
print(df)

爬取進度:   1%|▍                                                  | 1/125 [00:58<2:00:38, 58.38s/it]    

2024-1-2 完成 - 100 筆


爬取進度:   2%|▊                                                  | 2/125 [01:09<1:02:07, 30.30s/it]    

2024-1-9 完成 - 100 筆


爬取進度:   2%|█▎                                                   | 3/125 [01:19<43:24, 21.35s/it]    

...

2026-5-5 完成 - 100 筆


爬取進度:  99%|██████████████████████████████████████████████████▌| 124/125 [22:45<00:10, 10.64s/it]    

2026-5-12 完成 - 100 筆


爬取進度: 100%|███████████████████████████████████████████████████| 125/125 [22:55<00:00, 11.01s/it]    


2026-5-19 完成 - 100 筆

總共爬取 12500 筆資料,已儲存至 steam_top_sellers.csv

            date                                   title         price
0       2024-1-2                      NARAKA: BLADEPOINT          免費遊玩
1       2024-1-2                   Monster Hunter: World    NT$ 860.00
2       2024-1-2                         Winter Memories    NT$ 254.00
3       2024-1-2                                   柏德之門3  NT$ 1,599.00
4       2024-1-2                     PUBG: BATTLEGROUNDS          免費遊玩
...          ...                                     ...           ...
12495  2026-5-19                                   龍胤立志傳    NT$ 328.00
12496  2026-5-19                       Gray Zone Warfare    NT$ 891.00
12497  2026-5-19                    《NBA 2K26》名人堂通行證：第7季    NT$ 590.00
12498  2026-5-19         Devil May Cry 4 Special Edition    NT$ 140.00
12499  2026-5-19  扶她×偽娘 FUTA FUCKS FEMBOYS\n（已依您的偏好設定隱藏）    NT$ 202.00

[12500 rows x 3 columns]


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import os
import platform
from datetime import datetime


# 跨平台字型偵測
def detect_font():
    sys = platform.system()
    if sys == 'Darwin':
        return 'PingFang TC', ['PingFang TC', 'Heiti TC', 'Apple LiGothic', 'Arial Unicode MS']
    elif sys == 'Windows':
        return 'Microsoft JhengHei', ['Microsoft JhengHei', 'Microsoft YaHei', 'SimHei']
    else:
        return 'Noto Sans CJK TC', ['Noto Sans CJK TC', 'WenQuanYi Micro Hei', 'DejaVu Sans']


UI_FONT, MPL_FONT_LIST = detect_font()

# matplotlib 字型設定(避免中文方框)
matplotlib.rcParams['font.sans-serif'] = MPL_FONT_LIST
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 100


# 設定
CSV_FILENAME = 'steam_top_sellers.csv'
TARGET_COUNT = 100
FREE_KEYWORDS = ['免費', '免費遊玩', 'Free', 'Free to Play', 'F2P']


# Apple 風格配色
COLORS = {
    'bg_main': '#f5f5f7',          # Apple 招牌灰
    'bg_panel': '#ffffff',
    'bg_card': '#ffffff',
    'bg_sidebar': '#fbfbfd',       # 側邊欄略白
    'bg_subtle': '#fafafa',
    'bg_input': '#ffffff',
    'bg_hover': '#f5f5f7',
    'fg_primary': '#1d1d1f',       # Apple 文字色
    'fg_secondary': '#6e6e73',
    'fg_muted': '#86868b',
    'fg_link': '#0066cc',
    'accent': '#0071e3',           # Apple 藍
    'accent_dark': '#0058b3',
    'border': '#d2d2d7',           # Apple 邊框
    'border_subtle': '#e8e8ed',
    'divider': '#f0f0f2',
    'good': '#34c759',
    'warn': '#ff9500',
    'bad': '#ff3b30',
    'nav_active_bg': '#e8e8ed',
    'nav_active_fg': '#0071e3',
}

CHART = {
    'free': '#0071e3',
    'paid': '#ff9500',
    'up': '#34c759',
    'down': '#ff3b30',
    'grid': '#e8e8ed',
    'text': '#1d1d1f',
    'text_secondary': '#6e6e73',
    'bg': '#ffffff',
}


# ============================================
# 資料層
# ============================================

class DataManager:
    
    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.df = pd.DataFrame()
        self.load()
    
    def load(self):
        if not os.path.exists(self.csv_path):
            return False
        self.df = pd.read_csv(self.csv_path)
        if self.df.empty:
            return False
        if 'date' in self.df.columns:
            self.df['date_dt'] = pd.to_datetime(self.df['date'], errors='coerce')
        else:
            self.df['date'] = '未知'
            self.df['date_dt'] = pd.NaT
        if 'rank' not in self.df.columns:
            self.df['rank'] = self.df.groupby('date').cumcount() + 1
        if self.df['date_dt'].notna().any():
            self.df['year'] = self.df['date_dt'].dt.year.astype('Int64').astype(str)
        else:
            self.df['year'] = '未知'
        self.df['is_free'] = self.df['price'].apply(self._is_free)
        self.df = self.df.sort_values(['date_dt', 'rank']).reset_index(drop=True)
        return True
    
    @staticmethod
    def _is_free(price):
        if pd.isna(price):
            return True
        return str(price).strip() in FREE_KEYWORDS
    
    def get_years(self):
        return sorted([y for y in self.df['year'].dropna().unique() if y != '未知'])
    
    def get_dates_by_year(self, year):
        mask = self.df['year'] == year
        return list(self.df.loc[mask].sort_values('date_dt')['date'].unique())
    
    def all_dates_sorted(self):
        return list(self.df.sort_values('date_dt')['date'].unique())
    
    def week(self, date):
        return self.df[self.df['date'] == date].sort_values('rank')
    
    def year(self, y):
        return self.df[self.df['year'] == y]
    
    def top_games(self, data=None, n=100):
        d = self.df if data is None else data
        if d.empty:
            return pd.DataFrame()
        stats = d.groupby('title').agg(
            上榜次數=('rank', 'count'),
            平均名次=('rank', 'mean'),
            最佳名次=('rank', 'min'),
            首次上榜=('date_dt', 'min'),
            最後上榜=('date_dt', 'max'),
        ).reset_index()
        stats = stats.sort_values(
            by=['上榜次數', '平均名次'], ascending=[False, True]
        ).head(n).reset_index(drop=True)
        stats.insert(0, '排名', range(1, len(stats) + 1))
        return stats
    
    def longevity(self, n=50):
        stats = self.df.groupby('title').agg(
            首次=('date_dt', 'min'),
            最後=('date_dt', 'max'),
            上榜次數=('rank', 'count'),
            平均名次=('rank', 'mean'),
        ).reset_index()
        stats['跨度天數'] = (stats['最後'] - stats['首次']).dt.days
        stats = stats.sort_values('跨度天數', ascending=False).head(n).reset_index(drop=True)
        stats.insert(0, '排名', range(1, len(stats) + 1))
        return stats
    
    def top10_residents(self, n=30):
        top10 = self.df[self.df['rank'] <= 10]
        stats = top10.groupby('title').agg(
            進入TOP10次數=('rank', 'count'),
            平均名次=('rank', 'mean'),
            最佳名次=('rank', 'min'),
        ).reset_index()
        stats = stats.sort_values(
            ['進入TOP10次數', '平均名次'], ascending=[False, True]
        ).head(n).reset_index(drop=True)
        stats.insert(0, '排名', range(1, len(stats) + 1))
        return stats
    
    def yearly_breakdown(self):
        bd = self.df.groupby('year').agg(
            總筆數=('title', 'count'),
            免費=('is_free', 'sum'),
            獨特遊戲=('title', 'nunique'),
            涵蓋週數=('date', 'nunique'),
        ).reset_index()
        bd['付費'] = bd['總筆數'] - bd['免費']
        bd['免費pct'] = bd['免費'] / bd['總筆數'] * 100
        bd['付費pct'] = bd['付費'] / bd['總筆數'] * 100
        return bd
    
    def weekly_free_paid(self):
        result = self.df.groupby('date').agg(
            免費=('is_free', 'sum'),
            總數=('title', 'count'),
            date_dt=('date_dt', 'first'),
        ).reset_index()
        result['付費'] = result['總數'] - result['免費']
        return result.sort_values('date_dt')
    
    def new_entries(self, date):
        week_data = self.week(date)
        if week_data.empty:
            return pd.DataFrame()
        date_dt = week_data['date_dt'].iloc[0]
        history = self.df[self.df['date_dt'] < date_dt]['title'].unique()
        new = week_data[~week_data['title'].isin(history)]
        return new[['rank', 'title', 'price']]
    
    def top_movers(self, date):
        dates = self.all_dates_sorted()
        if date not in dates or dates.index(date) == 0:
            return pd.DataFrame(), pd.DataFrame()
        prev_date = dates[dates.index(date) - 1]
        now = self.week(date)[['title', 'rank']].set_index('title')
        prev = self.week(prev_date)[['title', 'rank']].set_index('title')
        merged = now.join(prev, lsuffix='_now', rsuffix='_prev').dropna()
        merged['變化'] = merged['rank_prev'] - merged['rank_now']
        merged = merged.reset_index()
        merged.columns = ['遊戲名稱', '本週名次', '上週名次', '變化']
        merged['上週名次'] = merged['上週名次'].astype(int)
        merged['本週名次'] = merged['本週名次'].astype(int)
        gainers = merged.sort_values('變化', ascending=False).head(15)
        losers = merged.sort_values('變化', ascending=True).head(15)
        return gainers, losers
    
    def game_timeline(self, title):
        return self.df[self.df['title'] == title].sort_values('date_dt')
    
    def search(self, keyword):
        return self.df[self.df['title'].str.contains(keyword, case=False, na=False)]
    
    def data_quality(self):
        counts = self.df.groupby('date').size().reset_index(name='筆數')
        counts['date_dt'] = pd.to_datetime(counts['date'])
        counts = counts.sort_values('date_dt')
        counts['狀態'] = counts['筆數'].apply(
            lambda x: '完整' if x >= TARGET_COUNT else ('不完整' if x >= 80 else '嚴重不足')
        )
        return counts[['date', '筆數', '狀態']]
    
    def price_changes(self):
        changes = []
        for title in self.df['title'].unique():
            game = self.df[self.df['title'] == title].sort_values('date_dt')
            if len(game) < 2:
                continue
            prev_free = None
            for _, row in game.iterrows():
                if prev_free is not None and prev_free != row['is_free']:
                    changes.append({
                        '遊戲名稱': title,
                        '日期': row['date'],
                        '變動': '付費 → 免費' if row['is_free'] else '免費 → 付費',
                    })
                prev_free = row['is_free']
        return pd.DataFrame(changes)


# ============================================
# UI 主程式 (Apple 風格)
# ============================================

class App:
    
    def __init__(self, root):
        self.root = root
        self.data = DataManager(CSV_FILENAME)
        
        if self.data.df.empty:
            messagebox.showerror("錯誤", f"找不到或無法讀取 {CSV_FILENAME}")
            self.root.destroy()
            return
        
        self._setup_window()
        self._setup_styles()
        self._build_layout()
        self.show_weekly()  # 預設進入週次分析
    
    def _setup_window(self):
        years = self.data.get_years()
        self.root.title(f"Steam Charts Analytics  ·  {years[0]} – {years[-1]}")
        self.root.geometry("1760x1020")
        self.root.minsize(1400, 820)
        self.root.configure(bg=COLORS['bg_main'])
    
    def _setup_styles(self):
        s = ttk.Style()
        s.theme_use('clam')
        
        # 通用
        s.configure('.', background=COLORS['bg_main'], foreground=COLORS['fg_primary'])
        
        # Frame
        s.configure('Main.TFrame', background=COLORS['bg_main'])
        s.configure('Card.TFrame', background=COLORS['bg_card'])
        s.configure('Sidebar.TFrame', background=COLORS['bg_sidebar'])
        
        # Label
        s.configure('TLabel', background=COLORS['bg_main'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 11))
        s.configure('Card.TLabel', background=COLORS['bg_card'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 11))
        s.configure('Title.TLabel', background=COLORS['bg_main'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 28, 'bold'))
        s.configure('Subtitle.TLabel', background=COLORS['bg_main'], foreground=COLORS['fg_secondary'],
                    font=(UI_FONT, 13))
        s.configure('CardTitle.TLabel', background=COLORS['bg_card'], foreground=COLORS['fg_secondary'],
                    font=(UI_FONT, 11))
        s.configure('CardValue.TLabel', background=COLORS['bg_card'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 26, 'bold'))
        s.configure('Section.TLabel', background=COLORS['bg_main'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 16, 'bold'))
        s.configure('FormLabel.TLabel', background=COLORS['bg_card'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 11))
        
        # 側邊欄
        s.configure('Sidebar.TLabel', background=COLORS['bg_sidebar'], foreground=COLORS['fg_primary'],
                    font=(UI_FONT, 11))
        s.configure('SidebarBrand.TLabel', background=COLORS['bg_sidebar'],
                    foreground=COLORS['fg_primary'], font=(UI_FONT, 17, 'bold'))
        s.configure('SidebarSub.TLabel', background=COLORS['bg_sidebar'],
                    foreground=COLORS['fg_secondary'], font=(UI_FONT, 10))
        s.configure('SidebarCat.TLabel', background=COLORS['bg_sidebar'],
                    foreground=COLORS['fg_muted'], font=(UI_FONT, 10, 'bold'))
        s.configure('SidebarStat.TLabel', background=COLORS['bg_sidebar'],
                    foreground=COLORS['fg_secondary'], font=(UI_FONT, 9))
        
        # Button - 主要 (Apple 藍)
        s.configure('TButton', background=COLORS['accent'], foreground='white',
                    borderwidth=0, focuscolor='none', padding=(20, 9),
                    font=(UI_FONT, 11, 'bold'))
        s.map('TButton', background=[('active', COLORS['accent_dark'])])
        
        # Button - 次要
        s.configure('Secondary.TButton', background=COLORS['bg_main'],
                    foreground=COLORS['fg_primary'], borderwidth=1,
                    bordercolor=COLORS['border'], focuscolor='none',
                    padding=(18, 8), font=(UI_FONT, 11))
        s.map('Secondary.TButton', background=[('active', COLORS['bg_hover'])])
        
        # Button - 導航
        s.configure('Nav.TButton', background=COLORS['bg_sidebar'],
                    foreground=COLORS['fg_primary'], borderwidth=0,
                    focuscolor='none', padding=(22, 12), anchor='w',
                    font=(UI_FONT, 11))
        s.map('Nav.TButton', background=[('active', COLORS['bg_hover'])])
        
        s.configure('NavActive.TButton', background=COLORS['nav_active_bg'],
                    foreground=COLORS['nav_active_fg'], borderwidth=0,
                    focuscolor='none', padding=(22, 12), anchor='w',
                    font=(UI_FONT, 11, 'bold'))
        
        # Combobox
        s.configure('TCombobox', fieldbackground=COLORS['bg_input'],
                    background=COLORS['bg_card'], foreground=COLORS['fg_primary'],
                    arrowcolor=COLORS['fg_primary'],
                    bordercolor=COLORS['border'], lightcolor=COLORS['border'],
                    darkcolor=COLORS['border'], borderwidth=1, padding=8,
                    font=(UI_FONT, 11))
        s.map('TCombobox', fieldbackground=[('readonly', COLORS['bg_input'])],
              foreground=[('readonly', COLORS['fg_primary'])],
              bordercolor=[('focus', COLORS['accent'])])
        
        # Entry
        s.configure('TEntry', fieldbackground=COLORS['bg_input'],
                    foreground=COLORS['fg_primary'], bordercolor=COLORS['border'],
                    borderwidth=1, padding=8, font=(UI_FONT, 11))
        s.map('TEntry', bordercolor=[('focus', COLORS['accent'])])
        
        # Notebook
        s.configure('TNotebook', background=COLORS['bg_main'], borderwidth=0,
                    tabmargins=[0, 5, 0, 0])
        s.configure('TNotebook.Tab', background=COLORS['bg_main'],
                    foreground=COLORS['fg_secondary'], padding=(22, 12),
                    font=(UI_FONT, 11), borderwidth=0)
        s.map('TNotebook.Tab',
              background=[('selected', COLORS['bg_card'])],
              foreground=[('selected', COLORS['accent'])],
              expand=[('selected', [0, 0, 0, 0])])
        
        # Treeview
        s.configure('Treeview', background=COLORS['bg_card'],
                    foreground=COLORS['fg_primary'],
                    fieldbackground=COLORS['bg_card'], borderwidth=0,
                    rowheight=36, font=(UI_FONT, 11))
        s.configure('Treeview.Heading', background=COLORS['bg_subtle'],
                    foreground=COLORS['fg_secondary'], borderwidth=0,
                    font=(UI_FONT, 10, 'bold'), padding=12)
        s.map('Treeview', background=[('selected', COLORS['accent'])],
              foreground=[('selected', 'white')])
        s.map('Treeview.Heading', background=[('active', COLORS['bg_hover'])])
        
        # Scrollbar
        s.configure('Vertical.TScrollbar', background=COLORS['bg_card'],
                    troughcolor=COLORS['bg_main'], borderwidth=0, arrowcolor=COLORS['fg_muted'])
        
        # Checkbutton
        s.configure('TCheckbutton', background=COLORS['bg_card'],
                    foreground=COLORS['fg_primary'], font=(UI_FONT, 11),
                    focuscolor='none')
    
    def _build_layout(self):
        # 側邊欄
        sidebar = tk.Frame(self.root, bg=COLORS['bg_sidebar'], width=260,
                           highlightbackground=COLORS['border_subtle'],
                           highlightthickness=0)
        sidebar.pack(side=tk.LEFT, fill=tk.Y)
        sidebar.pack_propagate(False)
        
        # 右側細分割線
        tk.Frame(sidebar, bg=COLORS['border_subtle'], width=1).place(
            relx=1.0, rely=0, relheight=1.0, x=-1)
        
        # 標題
        title_box = tk.Frame(sidebar, bg=COLORS['bg_sidebar'])
        title_box.pack(fill=tk.X, pady=(32, 8), padx=28)
        ttk.Label(title_box, text="Steam Charts", style='SidebarBrand.TLabel').pack(anchor='w')
        ttk.Label(title_box, text="Analytics Dashboard", style='SidebarSub.TLabel').pack(anchor='w')
        
        tk.Frame(sidebar, bg=COLORS['divider'], height=1).pack(fill=tk.X, padx=24, pady=20)
        
        # 導航(10 個頁面,刪除總覽儀表板)
        self.nav_buttons = {}
        nav_items = [
            ('核心分析', [
                ('weekly', '週次分析', self.show_weekly),
                ('yearly', '年度分析', self.show_yearly),
                ('trend', '趨勢分析', self.show_trend),
            ]),
            ('遊戲洞察', [
                ('rankings', '排行榜', self.show_rankings),
                ('detail', '遊戲查詢', self.show_game_query),
                ('movers', '名次變化', self.show_movers),
                ('newcomer', '新進榜', self.show_newcomers),
                ('priceChg', '價格變動', self.show_price_changes),
            ]),
            ('系統', [
                ('quality', '資料品質', self.show_quality),
                ('export', '匯出資料', self.show_export),
            ]),
        ]
        
        for cat_name, items in nav_items:
            ttk.Label(sidebar, text=cat_name, style='SidebarCat.TLabel').pack(
                anchor='w', padx=32, pady=(14, 6))
            for key, label, cmd in items:
                btn = ttk.Button(sidebar, text="   " + label,
                                 command=lambda k=key, c=cmd: self._switch(k, c),
                                 style='Nav.TButton')
                btn.pack(fill=tk.X, padx=14, pady=1)
                self.nav_buttons[key] = btn
        
        # 底部
        bottom = tk.Frame(sidebar, bg=COLORS['bg_sidebar'])
        bottom.pack(side=tk.BOTTOM, fill=tk.X, padx=28, pady=22)
        tk.Frame(bottom, bg=COLORS['divider'], height=1).pack(fill=tk.X, pady=(0, 14))
        self.status_label = ttk.Label(bottom, text=f"資料筆數  {len(self.data.df):,}",
                                      style='SidebarStat.TLabel')
        self.status_label.pack(anchor='w')
        ttk.Label(bottom, text=f"最後更新  {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                  style='SidebarStat.TLabel').pack(anchor='w', pady=(2, 12))
        ttk.Button(bottom, text="重新載入資料", command=self._reload,
                   style='Secondary.TButton').pack(fill=tk.X)
        
        # 主內容
        self.content = ttk.Frame(self.root, style='Main.TFrame')
        self.content.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=44, pady=36)
        self.current_view = None
    
    def _switch(self, key, cmd):
        for k, btn in self.nav_buttons.items():
            btn.configure(style='NavActive.TButton' if k == key else 'Nav.TButton')
        cmd()
    
    def _clear(self):
        for w in self.content.winfo_children():
            w.destroy()
    
    def _reload(self):
        self.data.load()
        self.status_label.configure(text=f"資料筆數  {len(self.data.df):,}")
        messagebox.showinfo("成功", f"已重新載入,共 {len(self.data.df):,} 筆")
        if self.current_view:
            self.current_view()
    
    def _header(self, title, subtitle=None):
        ttk.Label(self.content, text=title, style='Title.TLabel').pack(anchor='w')
        if subtitle:
            ttk.Label(self.content, text=subtitle, style='Subtitle.TLabel').pack(
                anchor='w', pady=(6, 28))
        else:
            ttk.Frame(self.content, style='Main.TFrame', height=28).pack(fill=tk.X)
    
    def _card(self, parent, **pack_kwargs):
        """產生 Apple 風格白色卡片(細邊框、無陰影)"""
        card = tk.Frame(parent, bg=COLORS['bg_card'],
                        highlightbackground=COLORS['border_subtle'],
                        highlightthickness=1)
        card.pack(**pack_kwargs)
        return card
    
    def _kpi(self, parent, col, label, value, color=None):
        card = tk.Frame(parent, bg=COLORS['bg_card'],
                        highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        card.grid(row=0, column=col, padx=6, sticky='nsew', ipady=14)
        parent.columnconfigure(col, weight=1)
        ttk.Label(card, text=label, style='CardTitle.TLabel').pack(anchor='w', padx=22, pady=(12, 4))
        lbl = ttk.Label(card, text=str(value), style='CardValue.TLabel')
        if color:
            lbl.configure(foreground=color)
        lbl.pack(anchor='w', padx=22, pady=(0, 12))
    
    def _control_bar(self):
        """控制列(白色卡片風格)"""
        bar = tk.Frame(self.content, bg=COLORS['bg_card'],
                       highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        bar.pack(fill=tk.X, pady=(0, 18), ipady=8, ipadx=18)
        return bar
    
    def _style_axes(self, ax, ylabel=None, xlabel=None, title=None):
        ax.set_facecolor(CHART['bg'])
        ax.tick_params(colors=CHART['text_secondary'], labelsize=10)
        for sp_name, sp in ax.spines.items():
            if sp_name in ('top', 'right'):
                sp.set_visible(False)
            else:
                sp.set_color(CHART['grid'])
                sp.set_linewidth(0.8)
        ax.grid(True, color=CHART['grid'], linestyle='-', alpha=0.6, linewidth=0.6)
        ax.set_axisbelow(True)
        if title:
            ax.set_title(title, fontsize=14, fontweight='bold', color=CHART['text'],
                         pad=18, loc='left')
        if xlabel:
            ax.set_xlabel(xlabel, fontsize=10, color=CHART['text_secondary'], labelpad=10)
        if ylabel:
            ax.set_ylabel(ylabel, fontsize=10, color=CHART['text_secondary'], labelpad=10)
    
    def _new_figure(self, figsize):
        """產生統一風格的 Figure(白底 + constrained_layout 解決尺度問題)"""
        fig = Figure(figsize=figsize, facecolor=CHART['bg'],
                     dpi=100, constrained_layout=True)
        return fig
    
    def _embed_fig(self, fig, parent):
        canvas = FigureCanvasTkAgg(fig, master=parent)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True, padx=16, pady=16)
    
    def _treeview(self, parent, columns, widths, data, row_tags=None, height=None):
        container = tk.Frame(parent, bg=COLORS['bg_card'])
        container.pack(fill=tk.BOTH, expand=True, padx=2, pady=2)
        kwargs = {'columns': columns, 'show': 'headings', 'style': 'Treeview'}
        if height:
            kwargs['height'] = height
        tree = ttk.Treeview(container, **kwargs)
        for col, w in zip(columns, widths):
            tree.heading(col, text=col, command=lambda c=col: self._sort_tree(tree, c, False))
            tree.column(col, width=w, anchor='center')
        tree.tag_configure('odd', background=COLORS['bg_card'])
        tree.tag_configure('even', background=COLORS['bg_subtle'])
        tree.tag_configure('good', background='#e8f7ec', foreground='#1d6e3a')
        tree.tag_configure('warn', background='#fff3e0', foreground='#7a4f00')
        tree.tag_configure('bad', background='#fde8e7', foreground='#8b1a13')
        for i, row in enumerate(data):
            tag = 'even' if i % 2 else 'odd'
            if row_tags and i < len(row_tags):
                tag = row_tags[i]
            tree.insert('', tk.END, values=row, tags=(tag,))
        vsb = ttk.Scrollbar(container, orient=tk.VERTICAL, command=tree.yview,
                            style='Vertical.TScrollbar')
        tree.configure(yscrollcommand=vsb.set)
        tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        vsb.pack(side=tk.RIGHT, fill=tk.Y)
        return tree
    
    def _sort_tree(self, tree, col, reverse):
        data = [(tree.set(k, col), k) for k in tree.get_children('')]
        try:
            data.sort(key=lambda x: float(x[0].replace('+', '').replace(',', '')), reverse=reverse)
        except ValueError:
            data.sort(reverse=reverse)
        for i, (_, k) in enumerate(data):
            tree.move(k, '', i)
        tree.heading(col, command=lambda: self._sort_tree(tree, col, not reverse))
    
    # ============================================
    # 1. 週次分析(單週 + 雙週比較)
    # ============================================
    def show_weekly(self):
        self.current_view = self.show_weekly
        self._clear()
        self._header("週次分析", "選一個日期看詳情,或啟用比較模式並排查看兩週")
        
        ctrl = self._control_bar()
        ttk.Label(ctrl, text="年份", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        year_var = tk.StringVar()
        year_cb = ttk.Combobox(ctrl, textvariable=year_var, values=self.data.get_years(),
                               width=8, state='readonly')
        year_cb.pack(side=tk.LEFT, padx=(0, 18))
        
        ttk.Label(ctrl, text="日期", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        date_var = tk.StringVar()
        date_cb = ttk.Combobox(ctrl, textvariable=date_var, width=15, state='readonly')
        date_cb.pack(side=tk.LEFT, padx=(0, 22))
        
        compare_var = tk.BooleanVar()
        chk = tk.Checkbutton(ctrl, text="啟用比較模式", variable=compare_var,
                             bg=COLORS['bg_card'], fg=COLORS['fg_primary'],
                             activebackground=COLORS['bg_card'],
                             selectcolor=COLORS['bg_card'],
                             font=(UI_FONT, 11), highlightthickness=0, bd=0)
        chk.pack(side=tk.LEFT, padx=(0, 18))
        
        date2_label = ttk.Label(ctrl, text="比較日期", background=COLORS['bg_card'],
                                font=(UI_FONT, 11))
        date2_var = tk.StringVar()
        date2_cb = ttk.Combobox(ctrl, textvariable=date2_var, width=15, state='readonly')
        
        def toggle_compare(*_):
            if compare_var.get():
                date2_label.pack(side=tk.LEFT, padx=(0, 8))
                date2_cb.pack(side=tk.LEFT, padx=(0, 18))
                date2_cb['values'] = self.data.all_dates_sorted()
            else:
                date2_label.pack_forget()
                date2_cb.pack_forget()
                date2_var.set('')
        compare_var.trace_add('write', toggle_compare)
        
        def on_year(*_):
            date_cb['values'] = self.data.get_dates_by_year(year_var.get())
            date_var.set('')
        year_cb.bind('<<ComboboxSelected>>', on_year)
        
        result = ttk.Frame(self.content, style='Main.TFrame')
        result.pack(fill=tk.BOTH, expand=True)
        
        def analyze():
            for w in result.winfo_children():
                w.destroy()
            date = date_var.get()
            if not date:
                messagebox.showwarning("提示", "請選擇日期")
                return
            if not compare_var.get():
                self._render_single_week(result, date)
            else:
                date2 = date2_var.get()
                if not date2:
                    messagebox.showwarning("提示", "請選擇比較日期")
                    return
                self._render_week_compare(result, date, date2)
        
        ttk.Button(ctrl, text="分析", command=analyze).pack(side=tk.LEFT)
    
    def _render_single_week(self, parent, date):
        data = self.data.week(date)
        free = int(data['is_free'].sum())
        paid = len(data) - free
        kpi = ttk.Frame(parent, style='Main.TFrame')
        kpi.pack(fill=tk.X, pady=(0, 18))
        for i, (l, v) in enumerate([
            ('當週筆數', len(data)),
            ('免費遊戲', free),
            ('付費遊戲', paid),
            ('免費比例', f"{free / len(data) * 100:.1f}%"),
        ]):
            self._kpi(kpi, i, l, v)
        body = ttk.Frame(parent, style='Main.TFrame')
        body.pack(fill=tk.BOTH, expand=True)
        
        # 左:圓餅卡片
        chart_card = tk.Frame(body, bg=COLORS['bg_card'], width=480,
                              highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        chart_card.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 12))
        chart_card.pack_propagate(False)
        fig = self._new_figure((4.6, 4.2))
        ax = fig.add_subplot(111)
        wedges, texts, autotexts = ax.pie(
            [free, paid], labels=['免費', '付費'],
            colors=[CHART['free'], CHART['paid']],
            autopct='%1.1f%%',
            textprops={'color': CHART['text'], 'fontsize': 11},
            wedgeprops={'edgecolor': 'white', 'linewidth': 4},
            pctdistance=0.75
        )
        for at in autotexts:
            at.set_color('white')
            at.set_fontweight('bold')
            at.set_fontsize(11)
        ax.set_title(date, fontsize=15, fontweight='bold',
                     color=CHART['text'], pad=20, loc='center')
        self._embed_fig(fig, chart_card)
        
        # 右:表格卡片
        table_card = tk.Frame(body, bg=COLORS['bg_card'],
                              highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table_card.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(12, 0))
        rows = [(r['rank'], r['title'], r['price']) for _, r in data.iterrows()]
        self._treeview(table_card, ['排名', '遊戲名稱', '價格'], [70, 420, 140], rows)
    
    def _render_week_compare(self, parent, date_a, date_b):
        da = self.data.week(date_a)
        db = self.data.week(date_b)
        fa, fb = int(da['is_free'].sum()), int(db['is_free'].sum())
        kpi = ttk.Frame(parent, style='Main.TFrame')
        kpi.pack(fill=tk.X, pady=(0, 18))
        for i, (l, v) in enumerate([
            (f'A 免費', fa), (f'A 付費', len(da) - fa),
            (f'B 免費', fb), (f'B 付費', len(db) - fb),
        ]):
            self._kpi(kpi, i, l, v)
        body = ttk.Frame(parent, style='Main.TFrame')
        body.pack(fill=tk.BOTH, expand=True)
        for col, (date, data) in enumerate([(date_a, da), (date_b, db)]):
            side = ttk.Frame(body, style='Main.TFrame')
            side.pack(side=tk.LEFT if col == 0 else tk.RIGHT,
                      fill=tk.BOTH, expand=True,
                      padx=(0, 12) if col == 0 else (12, 0))
            ttk.Label(side, text=date, style='Section.TLabel').pack(anchor='w', pady=(0, 10))
            tbl = tk.Frame(side, bg=COLORS['bg_card'],
                           highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            tbl.pack(fill=tk.BOTH, expand=True)
            rows = [(r['rank'], r['title'][:32], r['price']) for _, r in data.iterrows()]
            self._treeview(tbl, ['名次', '遊戲名稱', '價格'], [70, 320, 130], rows)
    
    # ============================================
    # 2. 年度分析
    # ============================================
    def show_yearly(self):
        self.current_view = self.show_yearly
        self._clear()
        self._header("年度分析", "所有年度橫向對比,並可選擇單一年度查看圓餅圖")
        
        ttk.Label(self.content, text="所有年度對比", style='Section.TLabel').pack(
            anchor='w', pady=(0, 12))
        table_card = tk.Frame(self.content, bg=COLORS['bg_card'],
                              highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table_card.pack(fill=tk.X, pady=(0, 28))
        compare = self.data.yearly_breakdown()
        rows = [(r['year'], r['涵蓋週數'], f"{r['總筆數']:,}", f"{r['獨特遊戲']:,}",
                 f"{int(r['免費'])}", f"{r['免費pct']:.1f}%", f"{r['付費pct']:.1f}%")
                for _, r in compare.iterrows()]
        self._treeview(table_card,
                       ['年份', '涵蓋週數', '總筆數', '獨特遊戲', '免費數', '免費%', '付費%'],
                       [110, 110, 140, 140, 110, 110, 110], rows,
                       height=min(len(rows), 6))
        
        ctrl = self._control_bar()
        ttk.Label(ctrl, text="查看單一年度", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 12))
        year_var = tk.StringVar()
        year_cb = ttk.Combobox(ctrl, textvariable=year_var, values=self.data.get_years(),
                               width=8, state='readonly')
        year_cb.pack(side=tk.LEFT, padx=(0, 18))
        
        result = ttk.Frame(self.content, style='Main.TFrame')
        result.pack(fill=tk.BOTH, expand=True)
        
        def analyze():
            for w in result.winfo_children():
                w.destroy()
            year = year_var.get()
            if not year:
                messagebox.showwarning("提示", "請選擇年份")
                return
            data = self.data.year(year)
            free = int(data['is_free'].sum())
            paid = len(data) - free
            chart = tk.Frame(result, bg=COLORS['bg_card'],
                             highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            chart.pack(fill=tk.BOTH, expand=True)
            fig = self._new_figure((8, 4.5))
            ax = fig.add_subplot(111)
            wedges, texts, autotexts = ax.pie(
                [free, paid], labels=['免費遊戲', '付費遊戲'],
                colors=[CHART['free'], CHART['paid']],
                autopct=lambda p: f'{p:.1f}%\n({int(p * len(data) / 100):,} 筆)',
                textprops={'color': CHART['text'], 'fontsize': 11},
                wedgeprops={'edgecolor': 'white', 'linewidth': 4},
                pctdistance=0.78, startangle=90
            )
            for at in autotexts:
                at.set_color('white')
                at.set_fontweight('bold')
                at.set_fontsize(11)
            ax.set_title(f'{year} 年度商業模式分布', fontsize=15, fontweight='bold',
                         color=CHART['text'], pad=20)
            self._embed_fig(fig, chart)
        
        ttk.Button(ctrl, text="顯示圓餅圖", command=analyze).pack(side=tk.LEFT)
    
    # ============================================
    # 3. 趨勢分析
    # ============================================
    def show_trend(self):
        self.current_view = self.show_trend
        self._clear()
        self._header("趨勢分析", "免費 vs 付費比例隨時間變化(年度粗略 / 每週細緻)")
        
        notebook = ttk.Notebook(self.content)
        notebook.pack(fill=tk.BOTH, expand=True)
        
        # Tab 1: 歷年
        tab1 = tk.Frame(notebook, bg=COLORS['bg_card'],
                        highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        notebook.add(tab1, text='   歷年趨勢   ')
        bd = self.data.yearly_breakdown()
        fig1 = self._new_figure((11, 5.5))
        ax = fig1.add_subplot(111)
        years = bd['year'].tolist()
        ax.plot(years, bd['免費pct'], 'o-', color=CHART['free'],
                label='免費', linewidth=2.5, markersize=10,
                markeredgecolor='white', markeredgewidth=2)
        ax.plot(years, bd['付費pct'], 'o-', color=CHART['paid'],
                label='付費', linewidth=2.5, markersize=10,
                markeredgecolor='white', markeredgewidth=2)
        ax.fill_between(years, bd['免費pct'], alpha=0.1, color=CHART['free'])
        ax.fill_between(years, bd['付費pct'], alpha=0.1, color=CHART['paid'])
        for i, (f, p) in enumerate(zip(bd['免費pct'], bd['付費pct'])):
            ax.annotate(f'{f:.1f}%', (years[i], f), textcoords="offset points",
                        xytext=(0, 14), ha='center', fontsize=10, fontweight='bold',
                        color=CHART['text'])
            ax.annotate(f'{p:.1f}%', (years[i], p), textcoords="offset points",
                        xytext=(0, -22), ha='center', fontsize=10, fontweight='bold',
                        color=CHART['text'])
        self._style_axes(ax, xlabel='年份', ylabel='百分比 (%)',
                         title='歷年免費 vs 付費比例')
        ax.legend(fontsize=11, loc='center right', frameon=False)
        ymin = min(bd['免費pct'].min(), bd['付費pct'].min()) - 12
        ymax = max(bd['免費pct'].max(), bd['付費pct'].max()) + 12
        ax.set_ylim(ymin, ymax)
        self._embed_fig(fig1, tab1)
        
        # Tab 2: 每週
        tab2 = tk.Frame(notebook, bg=COLORS['bg_card'],
                        highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        notebook.add(tab2, text='   每週走勢   ')
        weekly = self.data.weekly_free_paid()
        fig2 = self._new_figure((11, 5.5))
        ax = fig2.add_subplot(111)
        ax.plot(weekly['date_dt'], weekly['免費'], '-', color=CHART['free'],
                label='免費', linewidth=2)
        ax.plot(weekly['date_dt'], weekly['付費'], '-', color=CHART['paid'],
                label='付費', linewidth=2)
        ax.fill_between(weekly['date_dt'], weekly['免費'], alpha=0.15, color=CHART['free'])
        ax.fill_between(weekly['date_dt'], weekly['付費'], alpha=0.15, color=CHART['paid'])
        self._style_axes(ax, xlabel='日期', ylabel='遊戲數量',
                         title='每週免費 vs 付費筆數')
        ax.legend(fontsize=11, loc='center right', frameon=False)
        fig2.autofmt_xdate()
        self._embed_fig(fig2, tab2)
    
    # ============================================
    # 4. 排行榜(熱門 / 長青 / TOP10 常駐)
    # ============================================
    def show_rankings(self):
        self.current_view = self.show_rankings
        self._clear()
        self._header("排行榜", "三種視角的遊戲排行,點分頁切換")
        
        notebook = ttk.Notebook(self.content)
        notebook.pack(fill=tk.BOTH, expand=True)
        
        # Tab 1: 熱門
        tab1 = ttk.Frame(notebook, style='Main.TFrame')
        notebook.add(tab1, text='   熱門遊戲 · 上榜次數   ')
        ctrl1 = tk.Frame(tab1, bg=COLORS['bg_main'])
        ctrl1.pack(fill=tk.X, pady=(18, 12))
        tk.Label(ctrl1, text="範圍", bg=COLORS['bg_main'], fg=COLORS['fg_primary'],
                 font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        scope_var = tk.StringVar(value='全部年份')
        ttk.Combobox(ctrl1, textvariable=scope_var,
                     values=['全部年份'] + self.data.get_years(),
                     width=12, state='readonly').pack(side=tk.LEFT, padx=(0, 18))
        table1 = tk.Frame(tab1, bg=COLORS['bg_card'],
                          highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table1.pack(fill=tk.BOTH, expand=True)
        
        def refresh1():
            for w in table1.winfo_children():
                w.destroy()
            scope = scope_var.get()
            stats = (self.data.top_games(n=100) if scope == '全部年份'
                     else self.data.top_games(data=self.data.year(scope), n=100))
            rows = [(r['排名'], r['title'], r['上榜次數'], f"{r['平均名次']:.1f}",
                     r['最佳名次'],
                     r['首次上榜'].strftime('%Y-%m-%d') if pd.notna(r['首次上榜']) else '-',
                     r['最後上榜'].strftime('%Y-%m-%d') if pd.notna(r['最後上榜']) else '-')
                    for _, r in stats.iterrows()]
            self._treeview(table1, ['排名', '遊戲名稱', '上榜次數', '平均名次', '最佳名次',
                                     '首次上榜', '最後上榜'],
                           [70, 340, 100, 100, 100, 120, 120], rows)
        
        ttk.Button(ctrl1, text="重新查詢", command=refresh1).pack(side=tk.LEFT)
        refresh1()
        
        # Tab 2: 長青
        tab2 = ttk.Frame(notebook, style='Main.TFrame')
        notebook.add(tab2, text='   長青遊戲 · 上榜跨度   ')
        long_stats = self.data.longevity(n=50)
        table2 = tk.Frame(tab2, bg=COLORS['bg_card'],
                          highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table2.pack(fill=tk.BOTH, expand=True, pady=(18, 0))
        rows2 = [(r['排名'], r['title'], r['跨度天數'], r['上榜次數'], f"{r['平均名次']:.1f}",
                  r['首次'].strftime('%Y-%m-%d') if pd.notna(r['首次']) else '-',
                  r['最後'].strftime('%Y-%m-%d') if pd.notna(r['最後']) else '-')
                 for _, r in long_stats.iterrows()]
        self._treeview(table2, ['排名', '遊戲名稱', '跨度天數', '上榜次數', '平均名次',
                                 '首次上榜', '最後上榜'],
                       [70, 320, 110, 110, 110, 120, 120], rows2)
        
        # Tab 3: TOP 10 常駐
        tab3 = ttk.Frame(notebook, style='Main.TFrame')
        notebook.add(tab3, text='   TOP 10 常駐者   ')
        res_stats = self.data.top10_residents(n=30)
        table3 = tk.Frame(tab3, bg=COLORS['bg_card'],
                          highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table3.pack(fill=tk.BOTH, expand=True, pady=(18, 0))
        rows3 = [(r['排名'], r['title'], r['進入TOP10次數'],
                  f"{r['平均名次']:.1f}", r['最佳名次'])
                 for _, r in res_stats.iterrows()]
        self._treeview(table3, ['排名', '遊戲名稱', '進入TOP10次數', '平均名次', '最佳名次'],
                       [70, 380, 150, 120, 120], rows3)
    
    # ============================================
    # 5. 遊戲查詢
    # ============================================
    def show_game_query(self):
        self.current_view = self.show_game_query
        self._clear()
        self._header("遊戲查詢", "搜尋關鍵字,或選擇遊戲查看完整歷史與名次變化")
        
        ctrl = self._control_bar()
        ttk.Label(ctrl, text="遊戲名稱", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        title_var = tk.StringVar()
        all_titles = sorted(self.data.df['title'].unique())
        cb = ttk.Combobox(ctrl, textvariable=title_var, values=all_titles, width=44)
        cb.pack(side=tk.LEFT, padx=(0, 18))
        
        result = ttk.Frame(self.content, style='Main.TFrame')
        result.pack(fill=tk.BOTH, expand=True)
        
        def analyze():
            for w in result.winfo_children():
                w.destroy()
            kw = title_var.get().strip()
            if not kw:
                messagebox.showwarning("提示", "請輸入或選擇遊戲名稱")
                return
            exact = self.data.game_timeline(kw)
            if not exact.empty:
                self._render_game_detail(result, kw, exact)
                return
            matches = self.data.search(kw)
            if matches.empty:
                ttk.Label(result, text=f"找不到包含「{kw}」的遊戲",
                          style='Subtitle.TLabel').pack(pady=40)
                return
            stats = self.data.top_games(data=matches, n=50)
            ttk.Label(result,
                      text=f"找到 {len(stats)} 款遊戲(共 {len(matches)} 筆紀錄),點選任一列再按查詢可看詳情",
                      style='Subtitle.TLabel').pack(anchor='w', pady=(0, 14))
            tbl = tk.Frame(result, bg=COLORS['bg_card'],
                           highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            tbl.pack(fill=tk.BOTH, expand=True)
            rows = [(r['排名'], r['title'], r['上榜次數'], f"{r['平均名次']:.1f}",
                     r['最佳名次'],
                     r['首次上榜'].strftime('%Y-%m-%d') if pd.notna(r['首次上榜']) else '-',
                     r['最後上榜'].strftime('%Y-%m-%d') if pd.notna(r['最後上榜']) else '-')
                    for _, r in stats.iterrows()]
            tree = self._treeview(tbl,
                                  ['排名', '遊戲名稱', '上榜次數', '平均名次',
                                   '最佳名次', '首次上榜', '最後上榜'],
                                  [70, 340, 100, 100, 100, 120, 120], rows)
            def on_select(_):
                sel = tree.selection()
                if sel:
                    vals = tree.item(sel[0], 'values')
                    title_var.set(vals[1])
            tree.bind('<<TreeviewSelect>>', on_select)
        
        ttk.Button(ctrl, text="查詢", command=analyze).pack(side=tk.LEFT)
        cb.bind('<Return>', lambda e: analyze())
    
    def _render_game_detail(self, parent, title, tl):
        kpi = ttk.Frame(parent, style='Main.TFrame')
        kpi.pack(fill=tk.X, pady=(0, 18))
        for i, (l, v) in enumerate([
            ('上榜次數', len(tl)),
            ('平均名次', f"{tl['rank'].mean():.1f}"),
            ('最佳名次', tl['rank'].min()),
            ('最差名次', tl['rank'].max()),
            ('首次上榜', tl['date'].iloc[0]),
            ('最後上榜', tl['date'].iloc[-1]),
        ]):
            self._kpi(kpi, i, l, v)
        chart = tk.Frame(parent, bg=COLORS['bg_card'],
                         highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        chart.pack(fill=tk.BOTH, expand=True)
        fig = self._new_figure((11, 4.6))
        ax = fig.add_subplot(111)
        ax.plot(tl['date_dt'], tl['rank'], 'o-', color=CHART['free'],
                linewidth=2.5, markersize=7,
                markeredgecolor='white', markeredgewidth=1.5)
        ax.fill_between(tl['date_dt'], tl['rank'], 100, alpha=0.1, color=CHART['free'])
        ax.invert_yaxis()
        self._style_axes(ax, xlabel='日期', ylabel='名次',
                         title=f'{title}  ·  排名時間軸')
        ax.set_ylim(101, 0)
        fig.autofmt_xdate()
        self._embed_fig(fig, chart)
    
    # ============================================
    # 6. 名次變化
    # ============================================
    def show_movers(self):
        self.current_view = self.show_movers
        self._clear()
        self._header("名次變化", "與上週相比,名次上升或下降最多的遊戲")
        
        ctrl = self._control_bar()
        ttk.Label(ctrl, text="本週日期", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        date_var = tk.StringVar()
        date_cb = ttk.Combobox(ctrl, textvariable=date_var,
                               values=self.data.all_dates_sorted(),
                               width=15, state='readonly')
        date_cb.pack(side=tk.LEFT, padx=(0, 18))
        
        result = ttk.Frame(self.content, style='Main.TFrame')
        result.pack(fill=tk.BOTH, expand=True)
        
        def analyze():
            for w in result.winfo_children():
                w.destroy()
            date = date_var.get()
            if not date:
                messagebox.showwarning("提示", "請選擇日期")
                return
            g, l = self.data.top_movers(date)
            if g.empty:
                ttk.Label(result, text="該週為首週,無上週資料可比較",
                          style='Subtitle.TLabel').pack(pady=40)
                return
            body = ttk.Frame(result, style='Main.TFrame')
            body.pack(fill=tk.BOTH, expand=True)
            left = ttk.Frame(body, style='Main.TFrame')
            left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 12))
            ttk.Label(left, text="上升最快 TOP 15", style='Section.TLabel').pack(
                anchor='w', pady=(0, 10))
            t1 = tk.Frame(left, bg=COLORS['bg_card'],
                          highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            t1.pack(fill=tk.BOTH, expand=True)
            r1 = [(r['遊戲名稱'], r['上週名次'], r['本週名次'], f"+{int(r['變化'])}")
                  for _, r in g.iterrows()]
            self._treeview(t1, ['遊戲名稱', '上週名次', '本週名次', '變化'],
                           [260, 90, 90, 90], r1, row_tags=['good'] * len(r1))
            right = ttk.Frame(body, style='Main.TFrame')
            right.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(12, 0))
            ttk.Label(right, text="下降最多 TOP 15", style='Section.TLabel').pack(
                anchor='w', pady=(0, 10))
            t2 = tk.Frame(right, bg=COLORS['bg_card'],
                          highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            t2.pack(fill=tk.BOTH, expand=True)
            r2 = [(r['遊戲名稱'], r['上週名次'], r['本週名次'], f"{int(r['變化'])}")
                  for _, r in l.iterrows()]
            self._treeview(t2, ['遊戲名稱', '上週名次', '本週名次', '變化'],
                           [260, 90, 90, 90], r2, row_tags=['bad'] * len(r2))
        
        ttk.Button(ctrl, text="查詢", command=analyze).pack(side=tk.LEFT)
    
    # ============================================
    # 7. 新進榜
    # ============================================
    def show_newcomers(self):
        self.current_view = self.show_newcomers
        self._clear()
        self._header("新進榜", "找出該週首次出現在排行榜的新遊戲")
        
        ctrl = self._control_bar()
        ttk.Label(ctrl, text="日期", background=COLORS['bg_card'],
                  font=(UI_FONT, 11)).pack(side=tk.LEFT, padx=(0, 8))
        date_var = tk.StringVar()
        date_cb = ttk.Combobox(ctrl, textvariable=date_var,
                               values=self.data.all_dates_sorted(),
                               width=15, state='readonly')
        date_cb.pack(side=tk.LEFT, padx=(0, 18))
        result = ttk.Frame(self.content, style='Main.TFrame')
        result.pack(fill=tk.BOTH, expand=True)
        
        def analyze():
            for w in result.winfo_children():
                w.destroy()
            date = date_var.get()
            if not date:
                messagebox.showwarning("提示", "請選擇日期")
                return
            new = self.data.new_entries(date)
            if new.empty:
                ttk.Label(result, text="該週沒有新進榜遊戲",
                          style='Subtitle.TLabel').pack(pady=40)
                return
            ttk.Label(result, text=f"{date}  共有 {len(new)} 款新遊戲首次進榜",
                      style='Section.TLabel').pack(anchor='w', pady=(0, 14))
            table = tk.Frame(result, bg=COLORS['bg_card'],
                             highlightbackground=COLORS['border_subtle'], highlightthickness=1)
            table.pack(fill=tk.BOTH, expand=True)
            rows = [(r['rank'], r['title'], r['price']) for _, r in new.iterrows()]
            self._treeview(table, ['名次', '遊戲名稱', '價格'], [90, 420, 170], rows,
                           row_tags=['good'] * len(rows))
        
        ttk.Button(ctrl, text="查詢", command=analyze).pack(side=tk.LEFT)
    
    # ============================================
    # 8. 價格變動
    # ============================================
    def show_price_changes(self):
        self.current_view = self.show_price_changes
        self._clear()
        self._header("價格變動", "遊戲在期間內從免費變付費(或反之)的紀錄")
        
        changes = self.data.price_changes()
        if changes.empty:
            ttk.Label(self.content, text="沒有發現任何價格變動",
                      style='Subtitle.TLabel').pack(pady=40)
            return
        kpi = ttk.Frame(self.content, style='Main.TFrame')
        kpi.pack(fill=tk.X, pady=(0, 18))
        to_free = (changes['變動'] == '付費 → 免費').sum()
        to_paid = (changes['變動'] == '免費 → 付費').sum()
        self._kpi(kpi, 0, '總變動次數', len(changes))
        self._kpi(kpi, 1, '付費 → 免費', to_free, COLORS['good'])
        self._kpi(kpi, 2, '免費 → 付費', to_paid, COLORS['warn'])
        self._kpi(kpi, 3, '涉及遊戲數', changes['遊戲名稱'].nunique())
        table = tk.Frame(self.content, bg=COLORS['bg_card'],
                         highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table.pack(fill=tk.BOTH, expand=True)
        rows = [(r['遊戲名稱'], r['日期'], r['變動']) for _, r in changes.iterrows()]
        tags = ['good' if r['變動'] == '付費 → 免費' else 'warn' for _, r in changes.iterrows()]
        self._treeview(table, ['遊戲名稱', '日期', '變動'], [340, 160, 220], rows, row_tags=tags)
    
    # ============================================
    # 9. 資料品質
    # ============================================
    def show_quality(self):
        self.current_view = self.show_quality
        self._clear()
        self._header("資料品質", f"檢查每個日期是否達到 {TARGET_COUNT} 筆")
        q = self.data.data_quality()
        good = (q['筆數'] >= TARGET_COUNT).sum()
        warn = ((q['筆數'] >= 80) & (q['筆數'] < TARGET_COUNT)).sum()
        bad = (q['筆數'] < 80).sum()
        kpi = ttk.Frame(self.content, style='Main.TFrame')
        kpi.pack(fill=tk.X, pady=(0, 18))
        self._kpi(kpi, 0, '完整週數', good, COLORS['good'])
        self._kpi(kpi, 1, '不完整週數', warn, COLORS['warn'])
        self._kpi(kpi, 2, '嚴重不足週數', bad, COLORS['bad'])
        self._kpi(kpi, 3, '總週數', len(q))
        table = tk.Frame(self.content, bg=COLORS['bg_card'],
                         highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        table.pack(fill=tk.BOTH, expand=True)
        rows = [(r['date'], r['筆數'], r['狀態']) for _, r in q.iterrows()]
        tag_map = {'完整': 'odd', '不完整': 'warn', '嚴重不足': 'bad'}
        tags = [tag_map[r['狀態']] for _, r in q.iterrows()]
        self._treeview(table, ['日期', '筆數', '狀態'], [220, 160, 220], rows, row_tags=tags)
    
    # ============================================
    # 10. 匯出資料
    # ============================================
    def show_export(self):
        self.current_view = self.show_export
        self._clear()
        self._header("匯出資料", "把分析結果存成 Excel 或 CSV 檔案")
        
        info = tk.Frame(self.content, bg=COLORS['bg_card'],
                        highlightbackground=COLORS['border_subtle'], highlightthickness=1)
        info.pack(fill=tk.BOTH, expand=True)
        
        exports = [
            ("原始資料", "全部資料,完整欄位 (CSV)",
             lambda: self._export(self.data.df, 'raw')),
            ("熱門遊戲 TOP 100", "全期綜合排名 (Excel)",
             lambda: self._export(self.data.top_games(n=100), 'top100', xlsx=True)),
            ("年度統計", "各年度免費 vs 付費 (Excel)",
             lambda: self._export(self.data.yearly_breakdown(), 'yearly', xlsx=True)),
            ("長青遊戲", "上榜期間最久 TOP 50 (Excel)",
             lambda: self._export(self.data.longevity(n=50), 'longevity', xlsx=True)),
            ("TOP 10 常駐者", "進入 TOP 10 次數最多 (Excel)",
             lambda: self._export(self.data.top10_residents(n=30), 'top10res', xlsx=True)),
            ("價格變動紀錄", "免費 vs 付費 轉換清單 (Excel)",
             lambda: self._export(self.data.price_changes(), 'pricechg', xlsx=True)),
            ("資料品質報告", "每週筆數檢查結果 (Excel)",
             lambda: self._export(self.data.data_quality(), 'quality', xlsx=True)),
        ]
        for i, (title, desc, cmd) in enumerate(exports):
            row = tk.Frame(info, bg=COLORS['bg_card'])
            row.pack(fill=tk.X, padx=32, pady=16)
            txt = tk.Frame(row, bg=COLORS['bg_card'])
            txt.pack(side=tk.LEFT, fill=tk.X, expand=True)
            tk.Label(txt, text=title, bg=COLORS['bg_card'], fg=COLORS['fg_primary'],
                     font=(UI_FONT, 13, 'bold')).pack(anchor='w')
            tk.Label(txt, text=desc, bg=COLORS['bg_card'], fg=COLORS['fg_secondary'],
                     font=(UI_FONT, 11)).pack(anchor='w', pady=(2, 0))
            ttk.Button(row, text="下載", command=cmd).pack(side=tk.RIGHT, padx=10)
            if i < len(exports) - 1:
                tk.Frame(info, bg=COLORS['divider'], height=1).pack(fill=tk.X, padx=32)
    
    def _export(self, df, hint, xlsx=False):
        ext = '.xlsx' if xlsx else '.csv'
        types = [('Excel 檔案', '*.xlsx')] if xlsx else [('CSV 檔案', '*.csv')]
        fn = filedialog.asksaveasfilename(
            defaultextension=ext, filetypes=types,
            initialfile=f'steam_{hint}_{datetime.now().strftime("%Y%m%d")}{ext}'
        )
        if not fn:
            return
        export_df = df.drop(columns=['date_dt'], errors='ignore')
        if xlsx:
            export_df.to_excel(fn, index=False)
        else:
            export_df.to_csv(fn, index=False, encoding='utf-8-sig')
        messagebox.showinfo("成功", f"已匯出至:\n{fn}")


def main():
    root = tk.Tk()
    App(root)
    root.mainloop()


if __name__ == "__main__":
    main()

2026-05-24 00:08:23.711 python3[80669:12408103] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
2026-05-24 00:08:25.185 python3[80669:12408103] error messaging the mach port for IMKCFRunLoopWakeUpReliable
